# Week 2 Lab: Doubling Times — From Plastic to Pandemics
**SCIE1500 — Analytical Methods for Scientists | Semester 2, 2026**

> **Lab format:** Work in your group. Discuss answers before typing code.
> **Today's session:** ~2 hours | Ask your tutor when stuck.

---
## 🌍 Scientific Scenario: Growth and Decay Across Three Domains

![Comparing exponential growth and decay across three domains](../images/W2_doubling_times.svg)

Your team is preparing educational materials comparing **exponential growth** across three domains — to help policymakers understand why percentage growth rates can be deceptive and why early intervention matters.

| Domain | System | Behaviour |
|--------|--------|-----------|
| Environment | Global plastic production | 8.4% annual growth since 1950 |
| Biology | *E. coli* bacteria | Doubles every 20 minutes |
| Physics | Carbon-14 | Half-life 5,730 years (decay) |

> *All three follow the same mathematical structure — exponential functions — differing only in timescale and direction.*

---
## 🎯 Group Task & Learning Objectives

| Part | Time | You will be able to... |
|------|------|------|
| A | ~25 min | Build exponential models for all three scenarios, writing them in both $P_0 e^{kt}$ and $P_0 b^t$ form and converting between them |
| B | ~15 min | Calculate doubling times, derive the doubling-time formula, and verify the Rule of 70 |
| C | ~15 min | Use logarithms to find when thresholds are crossed, solving for unknown time in any exponential scenario |

### Before you start: loading the tools

The cell below loads the Python libraries we need today:
- **numpy** (`np`): handles arrays of numbers and mathematical operations
- **matplotlib** (`plt`): creates plots and graphs

Run it now — it sets up the plotting style for the rest of the lab. If you see an error, raise your hand.

In [ ]:
# Run this cell first — loads libraries for today's lab
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

---
## Part A: Exponential Models (~25 min)

The general exponential model is:
$$P(t) = P_0 \, e^{kt}, \qquad k > 0 \text{ (growth)}, \quad k < 0 \text{ (decay)}$$

An equivalent form is $P(t) = P_0 \, b^t$ where $b = e^k$. All three scenarios in today's lab use this same structure — they differ only in what $P_0$, $k$, and $t$ represent.

### A.1 — Plastic Production

We know plastic grew at **8.4% per year** since 1950, starting from 2 million tonnes. But that 8.4% is a *discrete* annual rate — our model needs a *continuous* rate $k$.

The conversion is: $k = \ln(1 + r)$, where $r$ is the decimal rate (0.084 here). Run the cell below, look at the raw value of $k$, then use it to predict production in 2025.

Plastic production: 8.4% annual growth, P0 = 2 M tonnes in 1950.

In [ ]:
# A.1 — Plastic production: 8.4% annual growth, P0 = 2 M tonnes in 1950
P0_plastic = 2        # million tonnes in 1950
r = 0.084             # 8.4% per year as a decimal

k_plastic = np.log(1 + r)           # k = ln(1 + r)
print("k_plastic =", k_plastic)     # raw value first

# How much plastic in 2025? t = 2025 - 1950 = 75 years
t_2025 = 2025 - 1950
P_2025 = P0_plastic * np.exp(k_plastic * t_2025)
print("Predicted 2025 production (million tonnes):", P_2025)
print("Actual ~500 M tonnes — is the model a good fit?")

✏️ **Activity A.1:**
1. In 2015 (t = 65 years), actual production was ~380 M tonnes. Evaluate the model at t = 65 and compare.
2. What does $k \approx 0.0807$ mean in plain English?

```python
# A.1 answers — fill in the blanks
P_2015 = P0_plastic * np.exp(k_plastic * 65)
print("Model at 2015:", P_2015)
# 1. Model vs actual:
#    ...
# 2. Interpretation of k:
#    ...
```

### A.2 — Bacterial Growth

For bacteria we're given the **doubling time** directly (every 20 minutes for *E. coli*). This lets us write the model in two equivalent forms:

- **Doubling form:** $N(t) = N_0 \times 2^{t/T_{\text{double}}}$ — reads naturally as "doubles every 20 min"
- **Continuous-rate form:** $N(t) = N_0 \, e^{kt}$ where $k = \ln(2)/T_{\text{double}}$

Both forms describe exactly the same population at every time $t$. The cell below runs both side by side so you can confirm they agree.

Bacterial growth: N0 = 1000 cells, doubles every 20 minutes.

In [ ]:
# A.2 — Bacterial growth: N0 = 1000 cells, doubles every 20 minutes
N0 = 1000
T_double = 20          # doubling time in minutes

k_bact = np.log(2) / T_double
print(f"k = ln(2)/20 = {k_bact:.4f} per minute")

def N_v1(t): return N0 * 2**(t / T_double)      # doubling form
def N_v2(t): return N0 * np.exp(k_bact * t)     # continuous-rate form

print()
print("Both forms should give identical counts:")
for t_hrs in [1, 2, 3]:
    t = t_hrs * 60
    print(f"  t = {t_hrs}h ({t} min):  doubling form = {N_v1(t):.0f}   k-form = {N_v2(t):.0f}")

✏️ **Activity A.2:**
1. How many cells after 3 hours? After how many minutes do you first exceed 1 million cells?
2. Starting from 1,000 cells, how many doublings are needed to reach 1 billion?

```python
# A.2 answers
# 1. Time to 1 million cells:
t_million = T_double * np.log2(1_000_000 / N0)
print("Minutes to 1 million:", t_million)
# 2. Doublings needed:
#    ...
```

### A.3 — Carbon-14 Decay

Radioactive decay follows the same exponential law as growth, but the quantity **shrinks** over time. The model is:
$$A(t) = A_0 \, e^{-\lambda t}$$
where $\lambda = \ln(2)/T_{\text{half}}$ is the *decay constant* — a positive number. The **negative sign in the exponent** is what makes the function decrease rather than increase.

For Carbon-14 (half-life 5,730 years): after each half-life, exactly 50% remains. We'll check this algebraically and verify with code.

Carbon-14 decay: A0 = 100 units, half-life = 5,730 years.

In [ ]:
# A.3 — Carbon-14 decay: A0 = 100 units, half-life = 5,730 years
A0_c14 = 100            # initial amount (treat as % of original)
T_half = 5730           # years

lambda_c14 = np.log(2) / T_half    # decay constant λ (positive)

def A_c14(t):
    return A0_c14 * np.exp(-lambda_c14 * t)    # negative exponent → decreasing

print(f"Decay constant λ = ln(2)/5730 = {lambda_c14:.6f} per year")
print()
print("Checking successive half-lives (should halve each time):")
for n in [1, 2, 3]:
    t = n * T_half
    print(f"  After {n} half-life(s) ({t:,} years):  A = {A_c14(t):.2f}   expected = {100 / 2**n:.2f}")

✏️ **Activity A.3:**
1. Confirm: after 2 half-lives, exactly 25% remains. Does the code agree?
2. Why is the exponent **negative** for decay but **positive** for growth?

```python
# A.3 answers:
# 1. After 2 half-lives:
print("After 2 half-lives:", A_c14(2 * T_half))
# 2. Why negative:
#    ...
```

---
## Part B: Doubling Time & the Rule of 70 (~15 min)

For any exponential growth model $P(t) = P_0 e^{kt}$, setting $P(T) = 2P_0$ and solving gives the **exact doubling time**:
$$T_{\text{double}} = \frac{\ln 2}{k}$$

For percentage rates (small $r$), this simplifies to the **Rule of 70**:
$$T_{\text{double}} \approx \frac{70}{r} \quad (r \text{ in percent})$$

The Rule of 70 is a useful mental shortcut — but how accurate is it, and when does it break down?

### B.1 — Comparing Exact Formula vs Rule of 70

We'll first check the Rule for our plastic scenario (8.4% growth), then scan a range of growth rates.

**Note on the table below:** The code uses column alignment specifiers like `>9` (right-align in 9 characters). You'll see these again in later labs — for now, just read the output and notice how error grows with rate.

How accurate is Rule of 70 for plastic growth (8.4% per year)?

In [ ]:
# B.1 — How accurate is Rule of 70 for plastic growth (8.4% per year)?
r_pct = 8.4

T_exact  = np.log(2) / k_plastic
T_rule70 = 70 / r_pct

print(f"Exact doubling time:      {T_exact:.2f} years")
print(f"Rule of 70 approximation: {T_rule70:.2f} years")
print(f"Error: {abs(T_exact - T_rule70) / T_exact * 100:.1f}%")

# Scan across a range of growth rates
print()
print(f"{'Rate':>8} | {'Exact (y)':>9} | {'Rule 70':>7} | {'Error':>6}")
print("-" * 40)
for r in [1, 2, 4, 8.4, 10, 15, 20]:
    T_e = np.log(2) / np.log(1 + r / 100)
    T_r = 70 / r
    print(f"{r:>7.1f}% | {T_e:>9.2f} | {T_r:>7.2f} | {abs(T_e - T_r) / T_e * 100:>5.1f}%")

✏️ **Activity B.1:**
1. At what growth rates is the Rule of 70 most accurate? When does it break down and why?
2. For bacteria doubling every 20 min, what is the **annual** equivalent percentage growth rate?
   *(Hint: 1 year ≈ 525,600 minutes. The per-minute k is already computed as `k_bact`.)*

```python
# B.1 answers:
# 1. Accuracy pattern:
#    ...
# 2. Annual rate for bacteria:
k_per_year = k_bact * 525600
annual_pct = (np.exp(k_per_year) - 1) * 100
print(f"Annual equivalent: {annual_pct:.2e}%")
```

### B.2 — Logarithmic Solving: When Are Thresholds Crossed?

Until now we've asked "how much at time t?" — plugging t into the model.
Now we flip the question: **"when does the population reach a given target?"**

Starting from $P_0 e^{kt} = C$, taking logarithms of both sides gives:
$$t = \frac{\ln(C / P_0)}{k}$$

We'll apply this to find when global plastic production crosses two major milestones.

When does plastic production reach 1 billion and 10 billion tonnes?

In [ ]:
# B.2 — When does plastic production reach 1 billion and 10 billion tonnes?
def years_to_reach(target_M):
    t = np.log(target_M / P0_plastic) / k_plastic
    return t, 1950 + t

t_1b,  yr_1b  = years_to_reach(1_000)     # 1 billion = 1,000 M tonnes
t_10b, yr_10b = years_to_reach(10_000)    # 10 billion = 10,000 M tonnes

print(f"1 billion tonnes:  t = {t_1b:.1f} years from 1950  →  year {yr_1b:.0f}")
print(f"10 billion tonnes: t = {t_10b:.1f} years from 1950  →  year {yr_10b:.0f}")
print()
print("Caveat: these projections assume constant 8.4% growth indefinitely.")
print("Discuss with your group: is that realistic?")

---
## Part C: Practice — Applying the Pattern (~15 min)

You now have all the tools: build the model, evaluate forward, or solve backwards with logarithms.
**Pattern to remember:** write model → set equal to target → take log → solve for t.

### C.1 — Invasive Carp

An invasive carp population is modelled as $P(t) = 50 \, e^{0.15t}$ (t in months).
Ecologists warn that once the population exceeds **1,000 fish**, manual eradication becomes infeasible.
We'll find when several thresholds are crossed, then plot the full trajectory to see the window for intervention.

Invasive carp: P(t) = 50 * e^(0.15t), t in months.

In [ ]:
# C.1 — Invasive carp: P(t) = 50 * e^(0.15t), t in months
P0_carp = 50
k_carp  = 0.15
limit   = 1000     # eradication becomes infeasible above this

def carp(t): return P0_carp * np.exp(k_carp * t)
def t_to_reach(target): return np.log(target / P0_carp) / k_carp

print("Time to reach each threshold:")
for target in [500, 1000, 5000]:
    t = t_to_reach(target)
    print(f"  {target:>5} fish:  {t:.1f} months  ({t / 12:.1f} years)")
        

The chart below shows the growth trajectory.

In [ ]:
# Plot the growth trajectory
t_vals = np.linspace(0, 55, 300)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_vals, carp(t_vals), "steelblue", lw=2)
ax.axhline(limit, color="red", ls="--", label=f"Eradication limit ({limit} fish)")
ax.axvline(t_to_reach(limit), color="orange", ls=":",
           label=f"Limit at t = {t_to_reach(limit):.0f} months")
ax.set_xlabel("Time (months)")
ax.set_ylabel("Carp population")
ax.set_title("Invasive Carp Population Growth")
ax.legend()
plt.tight_layout()
plt.show()

### C.2 — Carbon Dating

Carbon dating works *backwards* through the decay model: given the fraction of C-14 that remains, we solve for the sample's age. If fraction $f$ remains:
$$f \cdot A_0 = A_0 \, e^{-\lambda t} \implies t = \frac{-\ln(f)}{\lambda}$$

Note: the $A_0$ cancels — we only need the *fraction* remaining, not the original amount.

A fossil has 25% of its original C-14 remaining. How old is it?

In [ ]:
# C.2 — A fossil has 25% of its original C-14 remaining. How old is it?
fraction = 0.25

t_fossil = -np.log(fraction) / lambda_c14
print(f"25% remaining → age = {t_fossil:.0f} years")
print(f"That is {t_fossil / T_half:.2f} half-lives")

# Try your own value:
# my_fraction = 0.10        # 10% remaining
# print(f"At 10%: age = {-np.log(my_fraction) / lambda_c14:.0f} years")

---
## ✅ Lab Wrap-Up

Before finishing, confirm your group has:

- [ ] Built exponential models for all three scenarios (Part A)
- [ ] Verified both forms ($P_0 e^{kt}$ and $P_0 b^t$) give the same answer (A.2)
- [ ] Applied the Rule of 70 and noted when it breaks down (Part B)
- [ ] Solved for unknown time using logarithms (Parts B.2, C)
- [ ] **Groups 1 & 2:** Opened your assigned Presentation Brief (Lab A or Lab B) and started Part A

> 💡 **Key pattern:** *Write the model → set equal to target → take log → solve for t.* This three-step process works across every exponential scenario.